In [1]:
# Loads the autoreload extension
%load_ext autoreload

# Tells Jupyter to automatically reload every module every time you execute a cell
%autoreload 2

In [2]:
import sys
import os

# Putanja do glavnog foldera (prilagodi broj '..' u zavisnosti od toga koliko duboko si u folderima)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

# Dodajemo ga u Pythonovu listu putanja za pretragu
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Dodata putanja: {project_root}")

Dodata putanja: /home/jovyan/src_zipp


In [3]:
import torch
if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.25, device=0)
    print("GPU memorija je uspešno ograničena na 18%.")

GPU memorija je uspešno ograničena na 18%.


In [4]:
from src.training.trainerUNet import TrainingPipelineUNet
pipeline = TrainingPipelineUNet(16)
pipeline.train_unet(
    train_epochs=0,
    model_num=30
)
# print(pipeline.eval_model(model_num = 22))

Ukupno pesama: 448 | Za trening: 359 | Za test: 89
--- Model Architecture Test ---
Input Spectrogram Shape:  torch.Size([2, 1, 128, 216])
--- Model Architecture Test ---
Input Spectrogram Shape:  torch.Size([2, 1, 128, 216])
Output Spectrogram Shape: torch.Size([2, 1, 128, 216])
Output Mask Shape:        torch.Size([2, 1, 128, 216])
Test Passed: The model successfully output denoised spectrogram normalized coordinates per audio clip!
----------------------------------------


In [5]:
from src.training.modelClass import AudioUNet
model = AudioUNet().to(pipeline.device)
model.load_params(
    MODEL_DIR = pipeline.MODEL_DIR,
    model_num = 30
)

In [6]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import torch
# import os # Added to handle directory creation

# model.eval()

# with torch.no_grad():
#     noisy_spec, clean_spec, _ = next(iter(pipeline.test_dataloader))
#     clean_spec = clean_spec.to(pipeline.device)
#     noisy_spec = noisy_spec.to(pipeline.device)

#     out_spec, _ = model(noisy_spec)
#     print(out_spec.shape)

# # 1. Isolate the first item in the batch: index [1] 
# # (Note: index 1 means you are grabbing the SECOND song in the batch)
# # 2. Isolate the single channel dimension: index [0]
# # 3. Move it off the GPU and convert to a numpy matrix
# img_noisy = noisy_spec[1, 0].detach().cpu().numpy()
# img_clean = clean_spec[1, 0].detach().cpu().numpy()
# img_output = out_spec[1, 0].detach().cpu().numpy() 


In [7]:
# # Set up a Matplotlib figure with 1 row and 3 columns
# fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# # --- Plot 1: Noisy Input ---
# # PRO TIP: Add vmin=0, vmax=1 to these heatmaps if your data is normalized!
# sns.heatmap(img_noisy, ax=axes[0], cmap="magma", cbar=True)
# axes[0].set_title("Noisy Input Spectrogram", fontsize=14)
# axes[0].invert_yaxis()  
# axes[0].set_xlabel("Time Frames")
# axes[0].set_ylabel("Frequency Bins")

# # --- Plot 2: Clean Original ---
# sns.heatmap(img_clean, ax=axes[1], cmap="magma", cbar=True)
# axes[1].set_title("Clean Original Spectrogram", fontsize=14)
# axes[1].invert_yaxis()
# axes[1].set_xlabel("Time Frames")
# axes[1].set_ylabel("Frequency Bins")

# # --- Plot 3: Output Recreated ---
# sns.heatmap(img_output, ax=axes[2], cmap="magma", cbar=True)
# axes[2].set_title("Output Recreated Spectrogram", fontsize=14) 
# axes[2].invert_yaxis()                                         
# axes[2].set_xlabel("Time Frames")                              
# axes[2].set_ylabel("Frequency Bins")                           

# # Clean up layout
# plt.tight_layout()

# # ==========================================
# # RESULT SAVING LOGIC
# # ==========================================
# # 1. Create a directory to hold the images if it doesn't exist
# save_dir = "evaluation_plots"
# os.makedirs(save_dir, exist_ok=True)

# # 2. Define the path (you can make this dynamic with timestamps if you want)
# save_path = os.path.join(save_dir, "spectrogram_comparison_batch1.png")

# # 3. Save the figure (dpi=300 makes it high quality)
# plt.savefig(save_path, dpi=300, bbox_inches='tight')
# print(f"Plot successfully saved to: {save_path}")
# # ==========================================

# # Display the plot in the notebook
# plt.show()

In [8]:
# import torch
# import numpy as np
# import librosa
# import IPython.display as ipd

# def spectrogram_to_audio(spec_tensor, sr=22050, hop_length=512, n_fft=2048, 
#                          min_db=20.0, max_db=80.0, fmin=0.0, fmax=8000.0):
#     """
#     Converts a normalized [0, 1] U-Net Mel-Spectrogram back into playable audio,
#     strictly adhering to the RecognitionPipelineConfig limits.
#     """
#     # 1. Move to CPU and convert to numpy
#     if torch.is_tensor(spec_tensor):
#         spec = spec_tensor.detach().cpu().numpy()
#     else:
#         spec = spec_tensor
        
#     spec = spec.squeeze()
    
#     # 2. Reverse Normalization
#     spec_db = (spec * (max_db - min_db)) + min_db
#     spec_power = librosa.db_to_power(spec_db)
    
#     # 3. Griffin-Lim with STRICT Frequency Mapping
#     print("Reconstructing phase with correct fmax ceiling...")
#     waveform = librosa.feature.inverse.mel_to_audio(
#         spec_power, 
#         sr=sr, 
#         n_fft=n_fft, 
#         hop_length=hop_length,
#         fmin=fmin,      # Locked to 0.0
#         fmax=fmax       # Locked to 8000.0 (Fixes the pitch shift!)
#     )

#     # 3. CHECK THE EXACT LENGTH
#     exact_duration = len(waveform) / sr
#     print(f"Raw Griffin-Lim Output: {exact_duration:.4f} seconds ({len(waveform)} samples)")
    
#     return ipd.Audio(waveform, rate=sr)

In [9]:
# # Extract exactly what the Griffin-Lim algorithm needs from your config
# CONFIG_SR = 22050
# CONFIG_N_FFT = 2048
# CONFIG_HOP = 512

# print("--- Original Noisy Audio ---")
# display(spectrogram_to_audio(
#     noisy_spec[0], 
#     sr=CONFIG_SR, 
#     n_fft=CONFIG_N_FFT, 
#     hop_length=CONFIG_HOP, 
#     min_db=-80.0, 
#     max_db=-20.0
# ))

# print("--- Clean Target Audio ---")
# display(spectrogram_to_audio(
#     clean_spec[0], 
#     sr=CONFIG_SR, 
#     n_fft=CONFIG_N_FFT, 
#     hop_length=CONFIG_HOP, 
#     min_db=-80.0, 
#     max_db=-20.0
# ))

# print("--- Denoised Output Audio ---")
# display(spectrogram_to_audio(
#     out_spec[0], 
#     sr=CONFIG_SR, 
#     n_fft=CONFIG_N_FFT, 
#     hop_length=CONFIG_HOP, 
#     min_db=-80.0, 
#     max_db=-20.0
# ))



In [15]:
from src.training.evaluaator import EvaluatorUNetMath
evaluator = EvaluatorUNetMath()
evaluator.evaluate(
    model,
    pipeline.test_dataloader,
    pipeline.original_dataloader,
    pipeline.device,
    500
)
    


--- 1. Loading Compressed Database ---
Loaded compressed index with 256780 unique, valid hashes.

--- 2. Extracting Noisy Queries (TEST SET) ---


100%|██████████| 456/456 [01:03<00:00,  7.17it/s]


Testing 7281 noisy songs (total 5450711 hashes).

--- 3. Glasanje sa vremenskim poravnanje ---


Voting on matches: 100%|██████████| 5450711/5450711 [01:23<00:00, 65639.50it/s]



--- 4. Sortiranje Matches (Histogram Peak) ---


In [16]:
print(evaluator.generate_report_card([1,2,3,10]))

Total Queries Evaluated: 7281

--- Performance at K Thresholds ---
Threshold    | Top-K Accuracy  | MRR@K
-------------------------------------------------------
K = 1        | 97.32% (0.973)   | 0.973
K = 2        | 98.41% (0.984)   | 0.979
K = 3        | 98.68% (0.987)   | 0.980
K = 10       | 99.20% (0.992)   | 0.981
